# Docling parsing prototype

Extracts the text of a PDF from `data/` with [docling](https://github.com/docling-project/docling)
and writes it to `parsing/output/`, so the search step can work on plain text/markdown
instead of a PDF.

Run the cells top to bottom. The first conversion downloads docling's layout models
(a few hundred MB), so that cell is slow the first time and much faster afterwards.

## 1. Setup

In [ ]:
import time
from pathlib import Path

from docling.document_converter import DocumentConverter

# Resolve the repo root so this works whether the notebook is run from
# parsing/ (VS Code's default) or from the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

PDF_PATH = REPO_ROOT / "data" / "example" / "example.pdf"  # any PDF under data/
OUTPUT_DIR = REPO_ROOT / "parsing" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("repo root:", REPO_ROOT)
print("pdf      :", PDF_PATH.name, "| exists:", PDF_PATH.exists())
print("output   :", OUTPUT_DIR)

## 2. Convert the PDF

`DocumentConverter` runs docling's layout model over the PDF and returns a structured
document - headings, paragraphs, tables and so on - rather than a flat text dump.

In [ ]:
start = time.time()

converter = DocumentConverter()
result = converter.convert(PDF_PATH)
document = result.document

print(f"converted in {time.time() - start:.1f}s")
print("pages:", len(document.pages))

## 3. What did docling find?

A quick look at the structure. This matters for search: knowing where the headings
are lets us chunk the document by section instead of by an arbitrary character count.

In [ ]:
from collections import Counter

labels = Counter(item.label for item, _level in document.iterate_items())
for label, count in labels.most_common():
    print(f"{count:5d}  {label}")

## 4. Export and save

In [ ]:
markdown = document.export_to_markdown()
plain_text = document.export_to_text()

stem = PDF_PATH.stem
md_path = OUTPUT_DIR / f"{stem}.md"
txt_path = OUTPUT_DIR / f"{stem}.txt"

md_path.write_text(markdown, encoding="utf-8")
txt_path.write_text(plain_text, encoding="utf-8")

print(f"markdown: {len(markdown):>9,} chars -> {md_path.relative_to(REPO_ROOT)}")
print(f"text    : {len(plain_text):>9,} chars -> {txt_path.relative_to(REPO_ROOT)}")

## 5. Preview

In [ ]:
print(markdown[:2000])

## Notes / next steps

- Markdown keeps the heading structure, so it is the better input for chunked search;
  the `.txt` is there if a model just wants raw prose.
- `make parse` (`parsing/pdf_parser.py`) does this for every registered PDF, mirroring
  the `data/<collection>/` layout into `parsing/output/`.
- Docling caches its models after the first run, so repeat conversions are much faster.
- Use this notebook to try docling options before changing `pdf_parser.py`.